In [ ]:
!pip install fpdf2
# pillow is default installed on colab
# !pip install pillow

In [2]:
def save_response_content(target_url, destination):
    import requests
    session = requests.Session()
    response = session.get(target_url, stream=True)

    CHUNK_SIZE = 32768
    with open(destination, "wb") as f:
        for chunk in response.iter_content(CHUNK_SIZE):
            if chunk:  # filter out keep-alive new chunks
                f.write(chunk)

#### cv gen

##### config

In [11]:
target_url_base = "https://github.com/lkYu92393/python-snippet/raw/refs/heads/main/static/font/"
target_file_list = [
    "DejaVuSans-Bold.ttf", "DejaVuSans.ttf",
    # "DejaVuSans-Oblique.ttf", "DejaVuSerif-Bold.ttf", "DejaVuSerif.ttf"
]

for file in target_file_list:
  save_response_content(target_url_base + file, f"./{file}")

In [14]:
CONFIG = {
    # Page Settings
    'page': {
        'width': 210,
        'height': 297,
    },

    # Margins
    'margins': {
        'left': 15,
        'right': 15,
        'top': 15,
        'bottom': 15,
    },

    # Layout
    'layout': {
        'column_count': 3,
        'name_width_ratio': 0.65,
        'date_width_ratio': 0.35,
        'bullet_indent': 15,
    },

    # Colors
    'colors': {
        'primary': (0, 0, 0),
        'secondary': (60, 60, 60),
        'muted': (80, 80, 80),
        'light': (140, 140, 140),
        'line': (180, 180, 180),
    },

    # Fonts (base sizes)
    'fonts': {
        'name': 28,
        'section_title': 12,
        'job_title': 11,
        'default': 9,
    },

    # Spacing (base units)
    'spacing': {
        'tight': 2,
        'small': 4,
        'medium': 6,
        'large': 8,
    },

    # Mode: 'nested' (projects in work) or 'separate' (projects as own section)
    'mode': 'nested',
}

SECTION_STYLES = {
    'header': {
        'name': {'font_size': 'name', 'color': 'primary', 'height': 14, 'bold': True},
        'contact': {'font_size': 'default', 'color': 'muted', 'height': 5},
        'salary': {'font_size': 'default', 'color': 'primary', 'height': 5},
        'spacing_after': 'small',
    },

    'work_experience': {
        'title': {'font_size': 'section_title', 'color': 'primary', 'height': 7, 'bold': True},
        'job_title': {'font_size': 'job_title', 'color': 'primary', 'height': 6, 'bold': True},
        'project_title': {'font_size': 'default', 'color': 'secondary', 'height': 5},
        'date': {'font_size': 'default', 'color': 'light', 'height': 6},
        'description': {'font_size': 'default', 'color': 'primary', 'height': 5},
        'bullet': {'font_size': 'default', 'color': 'primary', 'height': 5},
        'spacing_after_job': 'small',
        'spacing_after_project': 'tight',
        'spacing_before_bullets': 'tight',
    },

    'projects': {
        'title': {'font_size': 'section_title', 'color': 'primary', 'height': 7, 'bold': True},
        'project_title': {'font_size': 'job_title', 'color': 'primary', 'height': 6, 'bold': True},
        'date': {'font_size': 'default', 'color': 'light', 'height': 6},
        'description': {'font_size': 'default', 'color': 'primary', 'height': 5},
        'bullet': {'font_size': 'default', 'color': 'primary', 'height': 5},
        'spacing_after_project': 'small',
        'spacing_before_bullets': 'tight',
    },

    'education': {
        'title': {'font_size': 'section_title', 'color': 'primary', 'height': 7, 'bold': True},
        'institution': {'font_size': 'job_title', 'color': 'primary', 'height': 6, 'bold': True},
        'degree': {'font_size': 'default', 'color': 'primary', 'height': 5},
        'date': {'font_size': 'default', 'color': 'light', 'height': 6},
        'spacing_after': 'small',
    },

    'languages': {
        'title': {'font_size': 'section_title', 'color': 'primary', 'height': 7, 'bold': True},
        'language': {'font_size': 'default', 'color': 'primary', 'height': 5, 'bold': True},
        'proficiency': {'font_size': 'default', 'color': 'primary', 'height': 5},
        'spacing_after': 'medium',
        'row_spacing': 12,
    },

    'skills': {
        'title': {'font_size': 'section_title', 'color': 'primary', 'height': 7, 'bold': True},
        'category': {'font_size': 'default', 'color': 'primary', 'height': 5, 'bold': True},
        'skills': {'font_size': 'default', 'color': 'primary', 'height': 5},
        'spacing_after': 'medium',
        'row_spacing': 12,
    },
}

LINE_STYLE = {
    'width': 0.25,
    'offset_y': 1,
}

SECTION_ORDER = {
    'nested': ['work_experience', 'education', 'skills', 'languages'],
    'separate': ['projects', 'work_experience', 'education', 'skills', 'languages'],
}

SECTION_TITLES = {
    'work_experience': 'WORK EXPERIENCE',
    'projects': 'PROJECTS',
    'education': 'EDUCATION',
    'languages': 'LANGUAGES',
    'skills': 'SKILLS',
}

FONT_FILES = {
    'regular': 'DejaVuSans.ttf',
    'bold': 'DejaVuSans-Bold.ttf',
}


In [7]:
class StyleResolver:
    """Resolves style references from CONFIG"""

    def __init__(self, config, section_styles):
        self.config = config
        self.section_styles = section_styles

    def get_font_size(self, size_key):
        if isinstance(size_key, int):
            return size_key
        return self.config['fonts'].get(size_key, self.config['fonts']['default'])

    def get_color(self, color_key):
        if isinstance(color_key, tuple):
            return color_key
        return self.config['colors'].get(color_key, self.config['colors']['primary'])

    def get_spacing(self, spacing_key):
        if isinstance(spacing_key, int):
            return spacing_key
        return self.config['spacing'].get(spacing_key, self.config['spacing']['medium'])

    def get_section_style(self, section_name):
        return self.section_styles.get(section_name, {})

    def get_computed_layout(self):
        available_width = (
            self.config['page']['width']
            - self.config['margins']['left']
            - self.config['margins']['right']
        )

        return {
            'available_width': available_width,
            'name_cell_width': available_width * self.config['layout']['name_width_ratio'],
            'date_cell_width': available_width * self.config['layout']['date_width_ratio'],
            'column_width': available_width / self.config['layout']['column_count'],
            'column_cell_width': (available_width / self.config['layout']['column_count']) - 5,
            'line_length': available_width,
            'bullet_indent': self.config['layout']['bullet_indent'],
        }

##### gen

In [15]:
import json
from fpdf import FPDF
import os

class CVGenerator(FPDF):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # Load fonts
        for style, filename in FONT_FILES.items():
            if os.path.exists(filename):
                self.add_font('DejaVu', 'B' if style == 'bold' else '', filename, uni=True)

    def header(self):
        pass

    def footer(self):
        pass

def extract_all_projects(jobs):
    """Extract all projects from jobs into a flat list (for separate mode)"""
    all_projects = []
    for job in jobs:
        if job.get('projects'):
            for project in job['projects']:
                # Add job context to project
                project_with_context = project.copy()
                project_with_context['job_position'] = job.get('position', '')
                project_with_context['job_company'] = job.get('company', '')
                # Use project date if available, otherwise use job date
                if not project_with_context.get('date'):
                    project_with_context['date'] = job.get('date', '')
                all_projects.append(project_with_context)
    return all_projects


def strip_projects_from_jobs(jobs):
    """Return jobs without projects (for separate mode)"""
    jobs_without_projects = []
    for job in jobs:
        job_copy = job.copy()
        job_copy.pop('projects', None)
        jobs_without_projects.append(job_copy)
    return jobs_without_projects


def render_header(pdf, contact, resolver, layout):
    """Render header section (name, contact, salary)"""
    style = resolver.get_section_style('header')

    # Name
    pdf.set_font('DejaVu', 'B', resolver.get_font_size(style['name']['font_size']))
    pdf.set_text_color(*resolver.get_color(style['name']['color']))
    pdf.cell(0, style['name']['height'], contact['name'], ln=True, align='C')

    # Contact
    pdf.set_font('DejaVu', '', resolver.get_font_size(style['contact']['font_size']))
    pdf.set_text_color(*resolver.get_color(style['contact']['color']))
    contact_line = f"{contact['email']}  |  {contact['phone']}  |  {contact['github']}"
    pdf.cell(0, style['contact']['height'], contact_line, ln=True, align='C')

    # Salary
    pdf.set_font('DejaVu', '', resolver.get_font_size(style['salary']['font_size']))
    pdf.set_text_color(*resolver.get_color(style['salary']['color']))
    salary_text = f"Current salary: {contact['current_salary']}, Expected salary: {contact['expected_salary']}, Available on {contact['available_date']}"
    pdf.cell(0, style['salary']['height'], salary_text, ln=True)

    pdf.ln(resolver.get_spacing(style['spacing_after']))


def render_section_title(pdf, title, resolver, layout):
    """Render section title with underline"""
    style = resolver.get_section_style('work_experience')['title']

    pdf.ln(resolver.get_spacing('large'))
    pdf.set_font('DejaVu', 'B', resolver.get_font_size(style['font_size']))
    pdf.set_text_color(*resolver.get_color(style['color']))
    pdf.cell(0, style['height'], title, ln=True)

    # Draw line
    x = pdf.get_x()
    y = pdf.get_y()
    pdf.set_line_width(LINE_STYLE['width'])
    pdf.set_draw_color(*resolver.get_color('line'))
    pdf.line(x, y + LINE_STYLE['offset_y'], x + layout['line_length'], y + LINE_STYLE['offset_y'])
    pdf.ln(resolver.get_spacing('medium'))


def render_work_experience(pdf, jobs, resolver, layout, include_projects=True):
    """Render work experience (optionally with projects)"""
    style = resolver.get_section_style('work_experience')

    for job in jobs:
        if not job.get('position', '').strip():
            continue

        job_title_text = f"{job['position']} - {job['company']}"
        pdf.set_font('DejaVu', 'B', resolver.get_font_size(style['job_title']['font_size']))
        pdf.set_text_color(*resolver.get_color(style['job_title']['color']))
        pdf.cell(layout['name_cell_width'], style['job_title']['height'], job_title_text, ln=False)

        pdf.set_font('DejaVu', '', resolver.get_font_size(style['date']['font_size']))
        pdf.set_text_color(*resolver.get_color(style['date']['color']))
        pdf.cell(layout['date_cell_width'], style['date']['height'], job['date'], ln=True, align='R')

        # Projects (only in nested mode)
        if include_projects and job.get('projects'):
            for project in job['projects']:
                if not project.get('name', '').strip():
                    continue

                pdf.ln(resolver.get_spacing(style['spacing_after_project']))

                pdf.set_font('DejaVu', '', resolver.get_font_size(style['project_title']['font_size']))
                pdf.set_text_color(*resolver.get_color(style['project_title']['color']))
                pdf.cell(0, style['project_title']['height'], project['name'], ln=True)

                pdf.set_font('DejaVu', '', resolver.get_font_size(style['description']['font_size']))
                pdf.set_text_color(*resolver.get_color(style['description']['color']))

                if project.get('tech_stack'):
                    full_text = f"Tech stack: {project['tech_stack']} {project['description']}"
                else:
                    full_text = project['description']
                pdf.multi_cell(0, style['description']['height'], full_text)

                if project.get('bullets'):
                    pdf.ln(resolver.get_spacing(style['spacing_before_bullets']))
                    for bullet in project['bullets']:
                        if bullet.strip():
                            pdf.set_font('DejaVu', '', resolver.get_font_size(style['bullet']['font_size']))
                            pdf.set_text_color(*resolver.get_color(style['bullet']['color']))
                            pdf.set_x(layout['bullet_indent'])
                            pdf.multi_cell(0, style['bullet']['height'], f"•  {bullet}")

        pdf.ln(resolver.get_spacing(style['spacing_after_job']))


def render_projects(pdf, projects, resolver, layout):
    """Render projects as separate section"""
    style = resolver.get_section_style('projects')

    for project in projects:
        if not project.get('name', '').strip():
            continue

        pdf.set_font('DejaVu', 'B', resolver.get_font_size(style['project_title']['font_size']))
        pdf.set_text_color(*resolver.get_color(style['project_title']['color']))
        pdf.cell(layout['name_cell_width'], style['project_title']['height'], project['name'], ln=False)

        pdf.set_font('DejaVu', '', resolver.get_font_size(style['date']['font_size']))
        pdf.set_text_color(*resolver.get_color(style['date']['color']))
        pdf.cell(layout['date_cell_width'], style['date']['height'], project.get('date', ''), ln=True, align='R')

        pdf.set_font('DejaVu', '', resolver.get_font_size(style['description']['font_size']))
        pdf.set_text_color(*resolver.get_color(style['description']['color']))

        if project.get('tech_stack'):
            full_text = f"Tech stack: {project['tech_stack']} {project['description']}"
        else:
            full_text = project['description']
        pdf.multi_cell(0, style['description']['height'], full_text)

        if project.get('bullets'):
            pdf.ln(resolver.get_spacing(style['spacing_before_bullets']))
            for bullet in project['bullets']:
                if bullet.strip():
                    pdf.set_font('DejaVu', '', resolver.get_font_size(style['bullet']['font_size']))
                    pdf.set_text_color(*resolver.get_color(style['bullet']['color']))
                    pdf.set_x(layout['bullet_indent'])
                    pdf.multi_cell(0, style['bullet']['height'], f"•  {bullet}")

        pdf.ln(resolver.get_spacing(style['spacing_after_project']))


def render_education(pdf, education, resolver, layout):
    """Render education section"""
    style = resolver.get_section_style('education')

    for edu in education:
        if not edu.get('institution', '').strip():
            continue

        pdf.set_font('DejaVu', 'B', resolver.get_font_size(style['institution']['font_size']))
        pdf.set_text_color(*resolver.get_color(style['institution']['color']))
        pdf.cell(layout['name_cell_width'], style['institution']['height'], edu['institution'], ln=False)

        pdf.set_font('DejaVu', '', resolver.get_font_size(style['date']['font_size']))
        pdf.set_text_color(*resolver.get_color(style['date']['color']))
        pdf.cell(layout['date_cell_width'], style['date']['height'], edu['date'], ln=True, align='R')

        pdf.set_font('DejaVu', '', resolver.get_font_size(style['degree']['font_size']))
        pdf.set_text_color(*resolver.get_color(style['degree']['color']))
        pdf.cell(0, style['degree']['height'], edu['degree'], ln=True)

        pdf.ln(resolver.get_spacing(style['spacing_after']))


def render_languages(pdf, languages, resolver, layout):
    """Render languages in columns"""
    style = resolver.get_section_style('languages')

    col_count = 0
    start_x = pdf.get_x()
    start_y = pdf.get_y()

    for lang in languages:
        if not lang.get('language', '').strip():
            continue

        x_pos = start_x + (col_count * layout['column_width'])
        pdf.set_xy(x_pos, start_y)

        font_style = 'B' if style['language'].get('bold', False) else ''
        pdf.set_font('DejaVu', font_style, resolver.get_font_size(style['language']['font_size']))
        pdf.set_text_color(*resolver.get_color(style['language']['color']))
        pdf.cell(layout['column_cell_width'], style['language']['height'], lang['language'], ln=False)

        pdf.set_xy(x_pos, start_y + style['language']['height'])
        pdf.set_font('DejaVu', '', resolver.get_font_size(style['proficiency']['font_size']))
        pdf.set_text_color(*resolver.get_color(style['proficiency']['color']))
        pdf.cell(layout['column_cell_width'], style['proficiency']['height'], lang['proficiency'], ln=False)

        col_count += 1
        if col_count >= CONFIG['layout']['column_count']:
            col_count = 0
            start_y += style['row_spacing']

    pdf.ln(resolver.get_spacing(style['spacing_after']))


def render_skills(pdf, skills, resolver, layout):
    """Render skills in columns"""
    style = resolver.get_section_style('skills')

    col_count = 0
    start_x = pdf.get_x()
    start_y = pdf.get_y()

    for skill in skills:
        if not skill.get('category', '').strip():
            continue

        x_pos = start_x + (col_count * layout['column_width'])
        pdf.set_xy(x_pos, start_y)

        font_style = 'B' if style['category'].get('bold', False) else ''
        pdf.set_font('DejaVu', font_style, resolver.get_font_size(style['category']['font_size']))
        pdf.set_text_color(*resolver.get_color(style['category']['color']))
        pdf.cell(layout['column_cell_width'], style['category']['height'], skill['category'], ln=False)

        pdf.set_xy(x_pos, start_y + style['category']['height'])
        pdf.set_font('DejaVu', '', resolver.get_font_size(style['skills']['font_size']))
        pdf.set_text_color(*resolver.get_color(style['skills']['color']))
        pdf.cell(layout['column_cell_width'], style['skills']['height'], skill['skills'], ln=False)

        col_count += 1
        if col_count >= CONFIG['layout']['column_count']:
            col_count = 0
            start_y += style['row_spacing']

    pdf.ln(resolver.get_spacing(style['spacing_after']))


def generate_cv(
    contact_info,
    jobs,
    education=None,
    languages=None,
    skills=None,
    mode='nested',
    section_order=None,
    output_filename=None
):
    """
    Generate CV PDF from a SINGLE source of truth

    Args:
        contact_info (dict): Name, email, phone, github, salary info
        jobs (list): Work experience with nested projects (SINGLE SOURCE)
        education (list): Education items
        languages (list): Language items
        skills (list): Skills items
        mode (str): 'nested' (projects in jobs) or 'separate' (projects as own section)
        section_order (list): Custom section order
        output_filename (str): Output filename (auto-generated if None)

    Returns:
        str: Path to generated PDF
    """
    # Defaults
    education = education or []
    languages = languages or []
    skills = skills or []

    # Set mode
    CONFIG['mode'] = mode.lower()

    # Get section order
    if section_order is None:
        section_order = SECTION_ORDER.get(CONFIG['mode'], SECTION_ORDER['nested'])

    # Prepare data based on mode (from single source)
    if CONFIG['mode'] == 'nested':
        # Use jobs as-is with projects
        jobs_to_render = jobs
        projects_to_render = []
    else:
        # Extract projects from jobs, strip from jobs
        jobs_to_render = strip_projects_from_jobs(jobs)
        projects_to_render = extract_all_projects(jobs)

    # Create PDF
    pdf = CVGenerator()
    pdf.add_page()
    pdf.set_auto_page_break(auto=True, margin=CONFIG['margins']['bottom'])
    pdf.set_margins(
        left=CONFIG['margins']['left'],
        top=CONFIG['margins']['top'],
        right=CONFIG['margins']['right']
    )

    # Initialize resolver
    resolver = StyleResolver(CONFIG, SECTION_STYLES)
    layout = resolver.get_computed_layout()

    # Render header
    render_header(pdf, contact_info, resolver, layout)

    # Define renderers
    renderers = {
        'work_experience': lambda: render_work_experience(
            pdf, jobs_to_render, resolver, layout,
            include_projects=(CONFIG['mode'] == 'nested')
        ),
        'projects': lambda: render_projects(pdf, projects_to_render, resolver, layout),
        'education': lambda: render_education(pdf, education, resolver, layout),
        'languages': lambda: render_languages(pdf, languages, resolver, layout),
        'skills': lambda: render_skills(pdf, skills, resolver, layout),
    }

    # Render sections in order
    for section_type in section_order:
        if section_type in renderers:
            title = SECTION_TITLES.get(section_type, section_type.upper())
            render_section_title(pdf, title, resolver, layout)
            renderers[section_type]()

    # Set metadata
    pdf.set_title(f"{contact_info['name']} - CV")
    pdf.set_author(contact_info['name'])
    pdf.set_subject('Curriculum Vitae')

    # Generate filename
    if output_filename is None:
        safe_name = contact_info['name'].replace(', ', '_').replace(' ', '_')
        output_filename = f"{safe_name}_CV.pdf"

    # Save
    pdf.output(output_filename)

    return output_filename

In [ ]:
contact = {
    "name": "NAME HERE",
    "email": "YOUR@EMAIL.XYZ",
    "phone": "12345678",
    "github": "github.com/link",
    "current_salary": 100000,
    "expected_salary": 200000,
    "available_date": "DD MMM YYYY"
}

jobs = [
    {
        "position": "Job 1",
        "company": "Company 1",
        "date": "Aug 2024 – Mar 2026",
        "projects": [
            {
                "name": "Project 1",
                "tech_stack": "C#, python, HTML5, JavaScript, MySQL",
                "description": "GOOD THING",
                "bullets": [
                    "Use of low code framework",
                    "Participated in designing implementation of features",
                    "Experience in dealing with users"
                ]
            }
        ]
    },
    {
        "position": "Job 2",
        "company": "Company 2",
        "date": "Jan 2022 – Aug 2024",
        "projects": [
            {
                "name": "Good Project",
                "tech_stack": "C#, HTML5, Javascript, SQLServer",
                "description": "Some more good things",
                "bullets": [
                    "Good people",
                    "Spearhead several features design and implementations"
                ]
            },
            {
                "name": "Good Project 2",
                "tech_stack": "ReactJS, NodeJS, Python",
                "description": "Something is done",
                "bullets": [
                    "NICE"
                ]
            }
        ]
    },
    {
        "position": "Job 3",
        "company": "Eternal Technology Consultant",
        "date": "Aug 2021 – Dec 2021",
        "projects": [
            {
                "name": "Another Project",
                "tech_stack": "C#, SQL Server",
                "description": "Good thing.",
                "bullets": []
            },
            {
                "name": "More Project",
                "tech_stack": "C#",
                "description": "More Good thing.",
                "bullets": []
            }
        ]
    }
]

# Education
education = [
    {
        "institution": "Some School",
        "degree": "More degree",
        "date": "Sep 2021 – Dec 2023"
    },
    {
        "institution": "Another School",
        "degree": "Bachelor degree",
        "date": "Sep 2012 – Aug 2015"
    }
]

# Languages
languages = [
    {"language": "English", "proficiency": "Good."},
    {"language": "Cantonese", "proficiency": "Native speaker"},
]

# Skills
skills = [
    {"category": "Slacking", "skills": "Epic"},
    {"category": "Making bad decision", "skills": "Legendary"},
]

# Generate CV (nested mode) - projects appear under each job
print("Generating CV (nested mode)...")
output = generate_cv(
    contact_info=contact,
    jobs=jobs,  # SAME data source
    education=education,
    languages=languages,
    skills=skills,
    mode='nested',
    output_filename='CV_Nested.pdf'
)
print(f"✅ Generated: {output}")

# Generate CV (separate mode) - projects extracted to own section
print("\nGenerating CV (separate mode)...")
output = generate_cv(
    contact_info=contact,
    jobs=jobs,  # SAME data source!
    education=education,
    languages=languages,
    skills=skills,
    mode='separate',
    output_filename='CV_Separate.pdf'
)
print(f"✅ Generated: {output}")

print("\n🎉 Done! Check your folder for the PDFs.")

#### PILLOW playground

In [24]:
# Requires: pip install pillow
def create_dummy_image(filename="dummy_image.jpg"):
    try:
        from PIL import Image, ImageDraw, ImageFont
        img = Image.new('RGB', (400, 300), color=(70, 130, 180))
        d = ImageDraw.Draw(img)
        d.text((10, 10), "Sample Image", fill=(255, 255, 255))
        img.save(filename)
        return filename
    except ImportError:
        print("Pillow not installed. Please provide an actual image file named 'dummy_image.jpg' or install Pillow.")
        # Fallback: Expect user to have the file
        return filename

def ensure_dummy_image(path: str, title: str = "Sample") -> str:
    """
    Create a dummy image if it doesn't exist.
    Requires: pip install pillow
    """
    if os.path.exists(path):
        return path

    try:
        from PIL import Image, ImageDraw, ImageFont
        img = Image.new('RGB', (800, 600), color=(70, 130, 180))
        d = ImageDraw.Draw(img)
        d.text((50, 50), title, fill=(255, 255, 255), font_size=40)
        img.save(path)
        print(f"Created dummy image: {path}")
        return path
    except ImportError:
        print("⚠️  Pillow not installed. Images will show placeholders.")
        return ""

#### GEN GRID LIKE PDF

In [35]:
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Any
from fpdf import FPDF
from fpdf.enums import Align, XPos, YPos
import os

# ==========================================
# 2. CONFIGURATION
# ==========================================
@dataclass
class FontConfig:
    name: int = 28
    section_title: int = 14
    default: int = 10

@dataclass
class ColorConfig:
    primary: tuple = (0, 0, 0)
    secondary: tuple = (60, 60, 60)
    light: tuple = (140, 140, 140)
    line: tuple = (180, 180, 180)

@dataclass
class LayoutConfig:
    column_count: int = 4
    cell_height: int = 15
    image_padding: int = 10  # Padding around image

@dataclass
class PDFConfig:
    DEFAULT_MARGINS = {'left': 15, 'right': 15, 'top': 15, 'bottom': 15}
    DEFAULT_SPACING = {'small': 4, 'medium': 6}

    page_width: int = 210
    page_height: int = 297
    margins: Dict[str, int] = field(default_factory=dict)
    spacing: Dict[str, int] = field(default_factory=dict)
    layout: LayoutConfig = field(default_factory=LayoutConfig)
    colors: ColorConfig = field(default_factory=ColorConfig)
    fonts: FontConfig = field(default_factory=FontConfig)

    @classmethod
    def from_dict(cls, config_dict: Dict[str, Any]) -> 'PDFConfig':
        layout = LayoutConfig(**config_dict.get('layout', {}))
        colors = ColorConfig(**config_dict.get('colors', {}))
        fonts = FontConfig(**config_dict.get('fonts', {}))

        return cls(
            page_width=config_dict.get('page', {}).get('width', 210),
            page_height=config_dict.get('page', {}).get('height', 297),
            margins=config_dict.get('margins', cls.DEFAULT_MARGINS),
            spacing=config_dict.get('spacing', cls.DEFAULT_SPACING),
            layout=layout,
            colors=colors,
            fonts=fonts,
        )

# ==========================================
# 3. DATA MODELS
# ==========================================
@dataclass
class Entry:
    id: str
    title: str
    summary: str
    image_path: Optional[str] = None

# ==========================================
# 4. PDF GENERATOR CLASS
# ==========================================
class GridPDF(FPDF):
    def __init__(self, config: PDFConfig):
        super().__init__(
            orientation="P",
            unit="mm",
            format=(config.page_width, config.page_height)
        )
        self.config = config
        self.set_margins(
            left=config.margins['left'],
            right=config.margins['right'],
            top=config.margins['top']
        )
        self.set_auto_page_break(auto=True, margin=config.margins['bottom'])
        self._link_ids: Dict[str, int] = {}

    def footer(self):
        self.set_y(-15)
        self.set_font('Helvetica', 'I', 8)
        self.set_text_color(*self.config.colors.light)
        self.cell(0, 10, f'Page {self.page_no()}', align='C')

    def add_toc_page(self, entries: List[Entry]):
        self.add_page()
        cfg = self.config

        self.set_font('Helvetica', 'B', cfg.fonts.section_title)
        self.set_text_color(*cfg.colors.primary)
        self.cell(0, 10, "Table of Contents", align='C', new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        self.ln(cfg.spacing['medium'])

        start_x = self.get_x()
        start_y = self.get_y()
        cols = cfg.layout.column_count

        usable_width = self.w - cfg.margins['left'] - cfg.margins['right']
        cell_width = usable_width / cols
        cell_height = cfg.layout.cell_height

        entries_on_page = 0

        for i, entry in enumerate(entries):
            row = entries_on_page // cols
            col = entries_on_page % cols

            x = start_x + (col * cell_width)
            y = start_y + (row * cell_height)

            if y > (cfg.page_height - 40):
                self.add_page()
                start_y = self.get_y() + 10
                entries_on_page = 0
                row = 0
                col = 0
                y = start_y
                x = start_x

            self.set_xy(x, y)
            self.set_font('Helvetica', 'B', cfg.fonts.default)
            self.set_draw_color(*cfg.colors.line)

            display_title = entry.title[:10]
            link_id = self.add_link()
            self._link_ids[entry.id] = link_id

            self.cell(cell_width, cell_height, display_title, border=1, align='C', link=link_id)

            entries_on_page += 1

    def add_detail_page(self, entry: Entry):
        self.add_page()
        cfg = self.config

        if entry.id in self._link_ids:
            link_id = self._link_ids[entry.id]
            self.set_link(link_id, page=self.page_no(), y=0)

        # Header
        self.set_font('Helvetica', 'B', cfg.fonts.name)
        self.set_text_color(*cfg.colors.primary)
        self.cell(0, 15, entry.title, align='C', new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        self.ln(cfg.spacing['small'])

        # Summary
        self.set_font('Helvetica', '', cfg.fonts.default)
        self.set_text_color(*cfg.colors.secondary)
        self.multi_cell(0, 10, entry.summary)
        self.ln(cfg.spacing['medium'])

        # ✅ Image with Full Width
        if entry.image_path and os.path.exists(entry.image_path):
            try:
                # Calculate available width - use full page width minus margins
                available_width = self.w - self.l_margin - self.r_margin

                # Alternative calculation if the above doesn't work:
                # available_width = cfg.page_width - cfg.margins['left'] - cfg.margins['right']

                # print(f"DEBUG: Page width={self.w}mm, Left margin={self.l_margin}mm, Right margin={self.r_margin}mm")
                # print(f"DEBUG: Available width for image={available_width}mm")

                # Save current position
                x_start = self.get_x()

                # Move to left margin
                self.set_x(cfg.margins['left'])

                # Insert image with full available width
                self.image(
                    entry.image_path,
                    x=self.get_x(),  # Explicit x position at left margin
                    y=None,
                    w=available_width  # Full width
                )

                # Move cursor down past the image
                self.set_y(self.get_y() + cfg.spacing['medium'])

                # print(f"✓ Image added: {entry.image_path} (width: {available_width}mm)")
            except Exception as e:
                self.set_text_color(255, 0, 0)
                self.cell(0, 10, f"Error loading image: {e}", align='C')
                self.set_text_color(*cfg.colors.secondary)
        else:
            # Placeholder if no image
            available_width = self.w - self.l_margin - self.r_margin
            self.set_fill_color(*cfg.colors.light)
            self.rect(
                self.l_margin,
                self.get_y(),
                available_width,
                50,
                style='F'
            )
            self.set_xy(self.l_margin, self.get_y() + 20)
            self.set_font('Helvetica', 'I', 10)
            self.set_text_color(*cfg.colors.secondary)
            self.cell(available_width, 10, "No Image Available", align='C')

    def generate_document(self, entries: List[Entry], output_path: str):
        self.add_toc_page(entries)
        for entry in entries:
            self.add_detail_page(entry)
        self.output(output_path)
        print(f"✓ Document saved to: {output_path}")


In [36]:
# Configuration
config_dict = {
    'page': {'width': 210, 'height': 297},
    'margins': {'left': 15, 'right': 15, 'top': 15, 'bottom': 15},
    'layout': {
        'column_count': 4,
        'cell_height': 15,
        'image_padding': 10,
    },
    'colors': {
        'primary': (0, 0, 0),
        'secondary': (60, 60, 60),
        'light': (140, 140, 140),
        'line': (180, 180, 180),
    },
    'fonts': {
        'name': 24,
        'section_title': 14,
        'default': 10,
    },
    'spacing': {
        'small': 4,
        'medium': 6,
    },
}

config = PDFConfig.from_dict(config_dict)

# ✅ Create dummy images for each entry
image_dir = "images"
os.makedirs(image_dir, exist_ok=True)

file_name = 'placeholder_img'

entries = [
    Entry(id="001", title="Alpha", summary="This is the summary for Alpha. It contains detailed information regarding the first item in our grid layout.",
            image_path=ensure_dummy_image(f"{image_dir}/{file_name}.jpg", "Alpha")),
    Entry(id="002", title="Beta", summary="Summary for Beta. This section covers the second item in our grid layout.",
            image_path=ensure_dummy_image(f"{image_dir}/{file_name}.jpg", "Beta")),
    Entry(id="003", title="Gamma", summary="Gamma details go here. A brief overview of the third component.",
            image_path=ensure_dummy_image(f"{image_dir}/{file_name}.jpg", "Gamma")),
    Entry(id="004", title="Delta", summary="Delta section summary. Explaining the fourth element of the dataset.",
            image_path=ensure_dummy_image(f"{image_dir}/{file_name}.jpg", "Delta")),
    Entry(id="005", title="Epsilon", summary="Epsilon info. The fifth item in the list with its specific details.",
            image_path=ensure_dummy_image(f"{image_dir}/{file_name}.jpg", "Epsilon")),
    Entry(id="006", title="Zeta", summary="Zeta summary text. Covering the sixth entry in the grid.",
            image_path=ensure_dummy_image(f"{image_dir}/{file_name}.jpg", "Zeta")),
]

image_path = ensure_dummy_image(f"{image_dir}/{file_name}.jpg", file_name)
entries_2 = [
    Entry(id=f"{i:03d}", title=f"Entry_{i}", summary=f"Details for entry {i}.", image_path=image_path)
    for i in range(1, 81)
]

# Generate
pdf = GridPDF(config)
pdf.generate_document(entries_2, "output_with_images.pdf")

✓ Document saved to: output_with_images.pdf


#### TEST